Script to map provided motif to the genome and create an output file based on the MEME FIMO output
Before running path to files should be adjusted

In [1]:
import pandas as pd
import numpy as np
from Bio import SeqIO
from Bio.Seq import Seq

import os

In [2]:
genome = '...\\genome_assembly\\SDOv20250616.fasta'

In [17]:
def find_all(seq, motif):
    # generator function to get all start positions of motif in the provided sequence
    start = seq.find(motif)

    while start != -1:
        yield start
        start = seq.find(motif, start + 1)


def detect_motif_coord(fasta_file, motif, motif_id, motif_alt_id, file_name=None):
    motif = motif.upper()
    rev_comp = str(Seq(motif).reverse_complement())
    motif_len = len(motif)

    results = []

    for record in SeqIO.parse(fasta_file, "fasta"):
        seq = str(record.seq).upper()

        # Forward strand check
        for start in find_all(seq, motif):
            results.append((
                motif_id,
                motif_alt_id,
                record.id,
                start + 1,                 # 1-based
                start + motif_len,
                "+"))

        # Reverse strand check
        for start in find_all(seq, rev_comp):
            results.append((
                motif_id,
                motif_alt_id,
                record.id,
                start + 1,
                start + motif_len,
                "-"))

    final_df = pd.DataFrame(results, columns=["motif_id", "motif_alt_id", "chrom", "start", "stop", "strand"])

    if file_name:
        final_df.to_csv(file_name, sep="\t", index=False)

    return final_df

def remove_overlap(tsv_fixed, tsv_check):
    motif_codes = np.arange(1, len(tsv_check) + 1, dtype=np.uint32)

    tsv_check.insert(len(tsv_check.columns), 'motif_code', motif_codes)

    merged = pd.merge(tsv_fixed, tsv_check, on='chrom', how='left', suffixes=('_fix', '_check'))

    overlap_condition = ((merged['start_check'] <= merged['stop_fix']) & (merged['stop_check'] >= merged['start_fix']))

    overlapping_rows = merged[overlap_condition]

    overlapping_check = overlapping_rows['motif_code'].unique()
    tsv_check_nooverlap = tsv_check[~tsv_check['motif_code'].isin(overlapping_check)]
    tsv_check_nooverlap = tsv_check_nooverlap.drop(tsv_check_nooverlap.columns[-1], axis=1)
    tsv_check.drop(columns=['motif_code'], inplace=True)
    return tsv_check_nooverlap

def fimo_tsv(fimo_path, motif_id):

    dtype_mapping = {
        'motif_id': 'object',
        'motif_alt_id': 'object',
        'sequence_name': 'object',
        'start': 'uint32',
        'stop': 'uint32',
        'strand': 'object'
    }

    fimo_path = fimo_path + '\\' + motif_id + "\\fimo.tsv"
    file_size = os.path.getsize(fimo_path)
    if file_size < 500:
        motif_tsv = pd.read_csv(fimo_path, sep='\t', skipfooter=3, engine='python')
        return motif_tsv
    else:
        motif_tsv = pd.read_csv(fimo_path, sep='\t', usecols=range(6), skipfooter=3,
                                engine='python', dtype=dtype_mapping)
        motif_tsv = motif_tsv.rename(columns={'sequence_name': 'chrom'}) #, 'start': 'start_m'})
        #######
        motif_tsv['chrom'] == motif_tsv["chrom"].str.replace("Sdo_chr", "", regex=True).astype('uint16')

        # motif_tsv = motif_tsv.drop(columns=['motif_id'])

        return motif_tsv

In [19]:
myc_df_gener = detect_motif_coord(fasta_file=genome,
                                   motif='CACGTG',
                                   motif_id='MA0147.4_fix',
                                   motif_alt_id="MYC_fix",
                                   file_name='')
print('CACGTG: ', len(myc_df_gener))

myc_noncan_1_df = detect_motif_coord(fasta_file=genome,
                                      motif='CCACGT',
                                      motif_id='MA0147.4_nc1',
                                      motif_alt_id="MYC_fix",
                                      file_name='')
print('CCACGT: ', len(myc_noncan_1_df))

myc_noncan_2_df = detect_motif_coord(fasta_file=genome,
                                      motif='ACGTGG',
                                      motif_id='MA0147.4_nc2',
                                      motif_alt_id="MYC_fix",
                                      file_name='')
print('ACGTGG: ', len(myc_noncan_2_df))

CACGTG:  28154
CCACGT:  18234
ACGTGG:  18234


Creating files combining coordinates of FIMO Myc and non-canonical Myc version

In [7]:
# read the fimo output for the motif
fimo_path = "...\\fimo_122_genome"
myc_fimo = fimo_tsv(fimo_path, 'MA0147.4')
print('fimo mapped myc motifs: ', len(myc_fimo))


fimo mapped myc motifs:  14123


In [15]:
# Combining fimo myc with exact matches of canonical motif
myc_fixed_nooverlap = remove_overlap(myc_fimo, myc_df_gener)
print('not overlapping mapped myc motifs: ', len(myc_fixed_nooverlap))
combined_df = pd.concat([myc_fimo, myc_fixed_nooverlap], ignore_index=True)
combined_df['sca_n'] = combined_df['chrom'].str.replace("Sdo_chr", "", regex=True).astype(int)
combined_sorted = combined_df.sort_values(by=['sca_n', 'start'])
combined_sorted = combined_sorted.drop(columns=['sca_n'])

combined_sorted['motif_alt_id'] = 'MYC_comb'
combined_sorted
combined_sorted['motif_id'] = 'MA0147.4_comb'

###
combined_sorted.to_csv('...\\MA0147.4_comb\\fimo.tsv', sep='\t', index=False)
combined_sorted


not overlapping mapped myc motifs:  9804


,motif_id,motif_alt_id,chrom,start,stop,strand
11512,MA0147.4_comb,MYC_comb,Sdo_chr1,10987,10994,+
5683,MA0147.4_comb,MYC_comb,Sdo_chr1,52799,52806,+
8683,MA0147.4_comb,MYC_comb,Sdo_chr1,71913,71920,+
8693,MA0147.4_comb,MYC_comb,Sdo_chr1,86925,86932,+
6886,MA0147.4_comb,MYC_comb,Sdo_chr1,94194,94201,-
...,...,...,...,...,...,...
23690,MA0147.4_comb,MYC_comb,Sdo_chr17,3928885,3928890,+
23926,MA0147.4_comb,MYC_comb,Sdo_chr17,3928885,3928890,-
669,MA0147.4_comb,MYC_comb,Sdo_chr17,3932354,3932361,-
11313,MA0147.4_comb,MYC_comb,Sdo_chr17,3932354,3932361,+


In [14]:
# Combining fimo myc with exact matches of canonical and non-canonical motifs
myc_fixed_nooverlap1 = remove_overlap(combined_sorted, myc_noncan_1_df)
myc_fixed_nooverlap2 = remove_overlap(myc_fixed_nooverlap1, myc_noncan_2_df)
print('not overlapping mapped myc motifs: ', len(myc_fixed_nooverlap2))
combined_df = pd.concat([myc_fimo, myc_fixed_nooverlap], ignore_index=True)
combined_df = pd.concat([combined_df, myc_fixed_nooverlap1], ignore_index=True)
combined_df = pd.concat([combined_df, myc_fixed_nooverlap2], ignore_index=True)
combined_df['sca_n'] = combined_df['chrom'].str.replace("Sdo_chr", "", regex=True).astype(int)
combined_sorted = combined_df.sort_values(by=['sca_n', 'start'])
combined_sorted = combined_sorted.drop(columns=['sca_n'])
combined_sorted['motif_alt_id'] = 'MYC_all'

combined_sorted['motif_id'] = 'MA0147.4_all'

####
combined_sorted.to_csv('...\\MA0147.4_comb_ext\\fimo.tsv', sep='\t', index=False)
combined_sorted


not overlapping mapped myc motifs:  18234


,motif_id,motif_alt_id,chrom,start,stop,strand
24831,MA0147.4_all,MYC_all,Sdo_chr1,7356,7361,-
11512,MA0147.4_all,MYC_all,Sdo_chr1,10987,10994,+
24832,MA0147.4_all,MYC_all,Sdo_chr1,24254,24259,-
24833,MA0147.4_all,MYC_all,Sdo_chr1,24445,24450,-
24834,MA0147.4_all,MYC_all,Sdo_chr1,24636,24641,-
...,...,...,...,...,...,...
23926,MA0147.4_all,MYC_all,Sdo_chr17,3928885,3928890,-
669,MA0147.4_all,MYC_all,Sdo_chr17,3932354,3932361,-
11313,MA0147.4_all,MYC_all,Sdo_chr17,3932354,3932361,+
41770,MA0147.4_all,MYC_all,Sdo_chr17,3932356,3932361,+


In [13]:
# Combining exact matches of canonical and non-canonical motifs
myc_fixed_nooverlap1 = remove_overlap(myc_df_gener, myc_noncan_1_df)
myc_fixed_nooverlap2 = remove_overlap(myc_fixed_nooverlap1, myc_noncan_2_df)
print('not overlapping mapped myc motifs: ', len(myc_fixed_nooverlap2))

combined_df = pd.concat([myc_df_gener, myc_fixed_nooverlap1], ignore_index=True)
combined_df = pd.concat([combined_df, myc_fixed_nooverlap2], ignore_index=True)
combined_df['sca_n'] = combined_df['chrom'].str.replace("Sdo_chr", "", regex=True).astype(int)
combined_sorted = combined_df.sort_values(by=['sca_n', 'start'])
combined_sorted = combined_sorted.drop(columns=['sca_n'])

combined_sorted['motif_alt_id'] = 'MYC_exact'

combined_sorted['motif_id'] = 'MA0147.4_exact'

###
combined_sorted.to_csv('...\\MA0147.4_exact\\fimo.tsv', sep='\t', index=False)
combined_sorted


not overlapping mapped myc motifs:  5688


,motif_id,motif_alt_id,chrom,start,stop,strand
28154,MA0147.4_exact,MYC_exact,Sdo_chr1,7356,7361,+
28155,MA0147.4_exact,MYC_exact,Sdo_chr1,24254,24259,+
28156,MA0147.4_exact,MYC_exact,Sdo_chr1,24445,24450,+
28157,MA0147.4_exact,MYC_exact,Sdo_chr1,24636,24641,+
28158,MA0147.4_exact,MYC_exact,Sdo_chr1,24827,24832,+
...,...,...,...,...,...,...
27492,MA0147.4_exact,MYC_exact,Sdo_chr17,3928885,3928890,+
28152,MA0147.4_exact,MYC_exact,Sdo_chr17,3928885,3928890,-
27493,MA0147.4_exact,MYC_exact,Sdo_chr17,3932355,3932360,+
28153,MA0147.4_exact,MYC_exact,Sdo_chr17,3932355,3932360,-
